## Tujuan

Tahap Modeling ini bertujuan untuk menerapkan algoritma Machine Learning pada data yang telah diproses. Proyek ini menggunakan dua pendekatan:
1. **Unsupervised Learning (Clustering)**: Menggunakan algoritma K-Means untuk membentuk segmentasi pelanggan berdasarkan pola perilakunya.
2. **Supervised Learning (Classification)**: Menggunakan Random Forest Classifier untuk memprediksi apakah pelanggan akan merespons kampanye pemasaran (variabel `Response`), dengan memanfaatkan fitur asli ditambah label cluster sebagai fitur prediktor baru.

In [13]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

Langkah pertama adalah memuat dataset hasil tahap preprocessing (`preprocessed_data.csv`). Data ini sudah dalam skala yang seragam (StandardScaler) dan siap digunakan untuk modeling.

In [14]:
# Memuat data
df = pd.read_csv('../data/preprocessed_data.csv')
print(f"[INFO] Data berhasil dimuat dengan {df.shape[0]} baris dan {df.shape[1]} kolom.")

# Memisahkan Fitur (X) dan Target (y)
X = df.drop(columns=['target'])
y = df['target']

# Train-Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"[INFO] Jumlah Data Latih (Train): {X_train.shape[0]}")
print(f"[INFO] Jumlah Data Uji (Test): {X_test.shape[0]}")

[INFO] Data berhasil dimuat dengan 1200 baris dan 22 kolom.
[INFO] Jumlah Data Latih (Train): 960
[INFO] Jumlah Data Uji (Test): 240


Kita menginisialisasi **Random Forest Classifier**. Parameter `class_weight='balanced'` ditambahkan agar model memberikan perhatian ekstra jika ada kelas target yang jumlah datanya lebih sedikit (minoritas).

In [15]:
# Inisialisasi Model Random Forest
rf_model = RandomForestClassifier(
    n_estimators=100, 
    random_state=42, 
    class_weight='balanced',
    n_jobs=-1 # Menggunakan seluruh core CPU agar training lebih cepat
)

# Melatih model menggunakan data training
print("[INFO] Sedang melatih model Random Forest...")
rf_model.fit(X_train, y_train)
print("[INFO] Model berhasil dilatih!")

[INFO] Sedang melatih model Random Forest...
[INFO] Model berhasil dilatih!


Sebelum menyimpan model, mari kita lakukan evaluasi singkat menggunakan data *Testing* untuk melihat seberapa akurat model kita dalam menebak status kesehatan mental remaja. Setelah itu, model diekspor ke dalam format `.pkl`.

In [16]:
# Melakukan prediksi pada data Testing
y_pred = rf_model.predict(X_test)

# Menghitung Akurasi
akurasi = accuracy_score(y_test, y_pred)
print(f"=== HASIL EVALUASI SINGKAT ===")
print(f"Akurasi Model: {akurasi * 100:.2f}%\n")

# Menampilkan Classification Report
target_names = ['Healthy (0)', 'Moderate (1)', 'At Risk (2)']
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=target_names))

# Menyimpan Model
model_path = '../models/classifier_model.pkl'
joblib.dump(rf_model, model_path)
print(f"[INFO] Model berhasil disimpan di: {model_path}")

=== HASIL EVALUASI SINGKAT ===
Akurasi Model: 100.00%

Classification Report:
              precision    recall  f1-score   support

 Healthy (0)       1.00      1.00      1.00        61
Moderate (1)       1.00      1.00      1.00       149
 At Risk (2)       1.00      1.00      1.00        30

    accuracy                           1.00       240
   macro avg       1.00      1.00      1.00       240
weighted avg       1.00      1.00      1.00       240

[INFO] Model berhasil disimpan di: ../models/classifier_model.pkl


Pada tahap ini, model prediksi kesehatan mental berbasis **Random Forest Classifier** telah berhasil dibangun dan dilatih. 
1. Data dibagi dengan proporsi **80:20** (Training:Testing).
2. Model berhasil memprediksi status kesejahteraan digital (*Healthy, Moderate, At Risk*) dengan metrik evaluasi awal yang dapat dilihat pada *Classification Report*.
3. Model akhir telah diekspor sebagai file `classifier_model.pkl` di dalam folder `/models`, siap untuk diintegrasikan dengan *StandardScaler* pada tahap **Deployment (Streamlit)**.